# scHiCAR → MERFISH MOp mapping — process & data structures

This notebook walks through how single-cell **scHiCAR** data is mapped onto the
**BICCN MERFISH** mouse primary motor cortex (MOp) spatial atlas with **Tangram**,
and shows the **input** and **output** data structure at every step.

```
INPUT                                   MAPPING                    OUTPUT
scHiCAR RNA (cells×genes) ─┐
scHiCAR 3D features        │  shared genes → Tangram               ad_map (scHiCAR×MERFISH prob.)
MERFISH RNA (cells×genes) ─┘  map_cells_to_space(cells,uniform) →  projected celltype / 3D in space
```
Note: Tangram aligns by **RNA only**; 3D features ride along via the mapping.


In [1]:
import logging, numpy as np, pandas as pd, scanpy as sc, tangram as tg
logging.getLogger().setLevel(logging.WARNING); sc.settings.verbosity = 0
import os
from pathlib import Path

def find_schicar_root():
    """$UCHROM_SCHICAR_ROOT, else example-data/schicar_mop (see example-data/README.md)."""
    if os.environ.get("UCHROM_SCHICAR_ROOT"):
        return Path(os.environ["UCHROM_SCHICAR_ROOT"])
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "example-data").is_dir():
            return p / "example-data" / "schicar_mop"
    raise FileNotFoundError("set UCHROM_SCHICAR_ROOT to the scHiCAR data directory")

ROOT = str(find_schicar_root())
print("scanpy", sc.__version__, "| tangram", tg.__version__)

scanpy 1.11.5 | tangram 1.0.4


## 1. INPUT A — scHiCAR RNA (single cell, no space)

In [2]:
adata_sch = sc.read_h5ad(f"{ROOT}/data/schicar/schicar_rna_mop.h5ad")
adata_sch

AnnData object with n_obs × n_vars = 5163 × 28783
    obs: 'celltype', 'DNAbarcode', 'nCount_RNA', 'nFeature_RNA', 'UMAP1', 'UMAP2'
    var: 'gene_id'
    obsm: 'X_umap'

In [3]:
print("shape (cells × genes):", adata_sch.shape)
print("obs columns:", list(adata_sch.obs.columns))
print("X:", type(adata_sch.X).__name__, adata_sch.X.dtype,
      "| raw counts? min/max =", float(adata_sch.X.min()), float(adata_sch.X.max()))
adata_sch.obs[["celltype", "DNAbarcode", "nCount_RNA", "nFeature_RNA"]].head()

shape (cells × genes): (5163, 28783)
obs columns: ['celltype', 'DNAbarcode', 'nCount_RNA', 'nFeature_RNA', 'UMAP1', 'UMAP2']
X: csr_matrix int64 | raw counts? min/max = 0.0 3492.0


,celltype,DNAbarcode,nCount_RNA,nFeature_RNA
ACGAACAAACTCAGACTT,Astro,ACGAACAAACTCAAGTCT,12792,3329
ACGAACAAACTCATTTGG,Sst,ACGAACAAACTCCCAAAT,9519,2705
ACGAACAAACTCCGTATT,L23IT.3,ACGAACAAACTCAATACG,14418,3285
ACGAACAAACTCGATAGG,Sst,ACGAACAAACTCCCTATC,28503,4522
ACGAACAAACTCTATCGC,Astro,ACGAACAAACTCGCGATA,8161,2413


In [4]:
# .var = genes (mouse symbols); .obs.celltype = the 19 cortical types
print("genes e.g.:", list(adata_sch.var_names[:8]))
adata_sch.obs["celltype"].value_counts().head(6)

genes e.g.: ['Gm26206', 'Xkr4', 'Gm18956', 'Gm1992', 'Gm7341', 'Gm10568', 'Gm37381', 'Rp1']


celltype
L5IT       642
L6CT       618
L23IT.1    575
Astro      407
L45IT      407
L6IT       373
Name: count, dtype: int64

## 2. INPUT B — MERFISH MOp (spatial, no 3D genome)

In [5]:
adata_mer = sc.read_h5ad(f"{ROOT}/data/merfish_mop/merfish_mouse2sample1.h5ad")
adata_mer

AnnData object with n_obs × n_vars = 13864 × 254
    obs: 'center_x', 'center_y', 'slice_id', 'sample_id', 'subclass', 'class_label'
    obsm: 'spatial'

In [6]:
print("shape (cells × genes):", adata_mer.shape, "  (254-gene panel)")
print("obs columns:", list(adata_mer.obs.columns))
print("obsm['spatial'] shape:", adata_mer.obsm["spatial"].shape, " = (x, y) centroids in µm")
adata_mer.obs[["subclass", "slice_id", "center_x", "center_y"]].head()

shape (cells × genes): (13864, 254)   (254-gene panel)
obs columns: ['center_x', 'center_y', 'slice_id', 'sample_id', 'subclass', 'class_label']
obsm['spatial'] shape: (13864, 2)  = (x, y) centroids in µm


,subclass,slice_id,center_x,center_y
index,,,,
100019614084014214413331344619872059390,L4/5 IT,mouse2_slice61,-4406.866048,-3784.263422
100024126192764500990464846159908804485,L4/5 IT,mouse2_slice1,-5938.214987,1334.835803
100036459431810739646600756615083690692,L4/5 IT,mouse2_slice300,1207.536506,2999.896805
100036464372885187596454120358701331853,L5 ET,mouse2_slice300,2121.430284,2490.716746
100042375868071099085129438220019302596,Oligo,mouse2_slice61,-4863.292523,-3079.352587


## 3. INPUT C — scHiCAR per-cell 3D-genome features (the payload to project)

In [7]:
hic = pd.read_csv(f"{ROOT}/data/schicar/hic_features.csv", index_col=0)
print("3D feature table shape (cells × features):", hic.shape)
print("index = scHiCAR RNA barcodes ; columns =", list(hic.columns))
hic.head()

3D feature table shape (cells × features): (5313, 8)
index = scHiCAR RNA barcodes ; columns = ['n_contacts', 'cis_frac', 'trans_frac', 'short_range_frac', 'long_range_frac', 'mean_log10_dist', 'pofs_slope', 'compartment_strength']


,n_contacts,cis_frac,trans_frac,short_range_frac,long_range_frac,mean_log10_dist,pofs_slope,compartment_strength
ACGAACAAACTCAGACTT,55403,0.747884,0.252116,0.710824,0.289176,5.899293,0.082503,0.903309
ACGAACAAACTCATTTGG,28122,0.606785,0.393215,0.804383,0.195617,5.669585,0.030094,0.925926
ACGAACAAACTCCGTATT,30234,0.744724,0.255276,0.820972,0.179028,5.540802,0.006347,0.932045
ACGAACAAACTCGATAGG,29638,0.703725,0.296275,0.874862,0.125138,5.370662,-0.038619,0.948406
ACGAACAAACTCTATCGC,25631,0.739651,0.260349,0.721120,0.278880,5.871076,0.075855,0.903576


## 4. Align inputs — shared genes
Tangram trains on genes present in **both** datasets (the bridge).

In [8]:
adata_sch.var_names = [g.lower() for g in adata_sch.var_names]; adata_sch.var_names_make_unique()
adata_mer.var_names = [g.lower() for g in adata_mer.var_names]; adata_mer.var_names_make_unique()
shared = sorted(set(adata_sch.var_names) & set(adata_mer.var_names))
print(f"shared genes: {len(shared)} / {adata_mer.n_vars} MERFISH panel")
print("examples:", shared[:12])

shared genes: 248 / 254 MERFISH panel
examples: ['1700022i11rik', '1810046k07rik', '5031425f14rik', '5730522e02rik', 'acta2', 'adam2', 'adamts2', 'adamts4', 'adra1b', 'alk', 'ankfn1', 'ano4']


## 5. Normalize + `tg.pp_adatas`

In [9]:
sc.pp.normalize_total(adata_sch, target_sum=1e4); sc.pp.log1p(adata_sch)
sc.pp.log1p(adata_mer)                      # MERFISH already volume-normalized
tg.pp_adatas(adata_sch, adata_mer, genes=shared)
print("training genes saved in .uns['training_genes']:", len(adata_sch.uns["training_genes"]))

training genes saved in .uns['training_genes']: 247


## 6. MAPPING — `tg.map_cells_to_space`
`mode='cells'`, `density_prior='uniform'` (single-cell MERFISH). `num_epochs=300`
here for a fast demo (production used 1000).

In [10]:
ad_map = tg.map_cells_to_space(
    adata_sch, adata_mer, mode="cells",
    density_prior="uniform", num_epochs=300, device="cpu")

Score: 0.443, KL reg: 0.000


Score: 0.879, KL reg: 0.005


Score: 0.889, KL reg: 0.004


## 7. OUTPUT ① — the mapping matrix `ad_map`
`ad_map` is an AnnData whose **rows = scHiCAR cells**, **columns = MERFISH cells**,
and `ad_map.X[i, j] = P(scHiCAR cell i is located at MERFISH cell j)`.

In [11]:
ad_map

AnnData object with n_obs × n_vars = 5163 × 13864
    obs: 'celltype', 'DNAbarcode', 'nCount_RNA', 'nFeature_RNA', 'UMAP1', 'UMAP2'
    var: 'center_x', 'center_y', 'slice_id', 'sample_id', 'subclass', 'class_label', 'uniform_density', 'rna_count_based_density'
    uns: 'train_genes_df', 'training_history'

In [12]:
M = ad_map.X
print("ad_map.shape (scHiCAR cells × MERFISH cells):", ad_map.shape)
print("M dtype:", M.dtype)
print("each scHiCAR cell's row sums to ~1 (a prob. distribution over space):",
      np.round(M.sum(1)[:5], 3))
print("a 3×5 corner of M:\n", np.round(M[:3, :5], 5))
print(f"max prob per scHiCAR cell: median {np.median(M.max(1)):.4f} "
      f"(uniform baseline = {1/ad_map.n_vars:.2e})")

ad_map.shape (scHiCAR cells × MERFISH cells): (5163, 13864)
M dtype: float32
each scHiCAR cell's row sums to ~1 (a prob. distribution over space): [1. 1. 1. 1. 1.]
a 3×5 corner of M:
 [[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
max prob per scHiCAR cell: median 0.0387 (uniform baseline = 7.21e-05)


## 8. OUTPUT ② — project cell-type labels into space
`project_cell_annotations` pushes any scHiCAR per-cell label through `M` to give,
for every MERFISH cell, a distribution over scHiCAR cell types.

In [13]:
tg.project_cell_annotations(ad_map, adata_mer, annotation="celltype")
pred = adata_mer.obsm["tangram_ct_pred"]
print("obsm['tangram_ct_pred'] shape (MERFISH cells × scHiCAR celltypes):", pred.shape)
adata_mer.obs["pred_celltype"] = pred.idxmax(axis=1).values
adata_mer.obs[["subclass", "pred_celltype"]].head()

obsm['tangram_ct_pred'] shape (MERFISH cells × scHiCAR celltypes): (13864, 19)


,subclass,pred_celltype
index,,
100019614084014214413331344619872059390,L4/5 IT,L45IT
100024126192764500990464846159908804485,L4/5 IT,L5IT
100036459431810739646600756615083690692,L4/5 IT,L45IT
100036464372885187596454120358701331853,L5 ET,PT
100042375868071099085129438220019302596,Oligo,Oligo


## 9. OUTPUT ③ — project the 3D-genome features into space
The same matrix carries the per-cell 3D features:
`spatial[j] = Σ_i M[i,j]·feature[i] / Σ_i M[i,j]` (weighted average per MERFISH cell).

In [14]:
F = hic.reindex(ad_map.obs_names)                 # align to scHiCAR-cell axis of M
nan = F.isna().to_numpy()
num = M.T @ np.where(nan, 0.0, F.to_numpy())
den = M.T @ (~nan).astype(float)
proj = pd.DataFrame(num / den, index=adata_mer.obs_names,
                    columns=["hic_" + c for c in F.columns])
print("projected 3D features shape (MERFISH cells × features):", proj.shape)
proj.head()

projected 3D features shape (MERFISH cells × features): (13864, 8)


,hic_n_contacts,hic_cis_frac,hic_trans_frac,hic_short_range_frac,hic_long_range_frac,hic_mean_log10_dist,hic_pofs_slope,hic_compartment_strength
index,,,,,,,,
100019614084014214413331344619872059390,23663.763155,0.733097,0.266903,0.775393,0.224607,5.756903,0.057016,0.918479
100024126192764500990464846159908804485,24434.232399,0.757417,0.242583,0.796762,0.203238,5.686185,0.042717,0.923470
100036459431810739646600756615083690692,23153.360673,0.729328,0.270672,0.779134,0.220866,5.766756,0.059626,0.917145
100036464372885187596454120358701331853,27633.560813,0.743273,0.256727,0.783694,0.216306,5.730820,0.051385,0.919412
100042375868071099085129438220019302596,19429.267361,0.747881,0.252119,0.714324,0.285676,5.815000,0.065028,0.907561


## 10. INPUT → OUTPUT data-structure summary

| stage | object | shape | meaning |
|---|---|---|---|
| input A | `adata_sch` (AnnData) | cells × genes | scHiCAR RNA + `obs.celltype/DNAbarcode` |
| input B | `adata_mer` (AnnData) | cells × 254 | MERFISH RNA + `obsm['spatial']`, `obs.subclass` |
| input C | `hic` (DataFrame) | cells × 8 | scHiCAR per-cell 3D features |
| mapping | `ad_map` (AnnData) | **scHiCAR × MERFISH** | `X[i,j]=P(cell i → space j)` |
| output ② | `obsm['tangram_ct_pred']` | MERFISH × celltypes | projected cell-type probs |
| output ③ | `proj` (DataFrame) | **MERFISH × 8** | 3D features placed in space |

**Key point:** `ad_map` is learned from **RNA only**; cell types and 3D features
are projected *afterwards* through it. So anything placed in space is determined
by the RNA bridge (see project README / validation for what this does and does
not license scientifically).

---
## 11. Other omic in space ① — imputed RNA transcriptome
`tg.project_genes` pushes the **full** scHiCAR transcriptome through the same map
to predict spatial expression of genes (here top 2000 HVGs) — including genes not
on the MERFISH panel.

In [15]:
sch_full = sc.read_h5ad(f"{ROOT}/data/schicar/schicar_rna_mop.h5ad")
sc.pp.normalize_total(sch_full, target_sum=1e4); sc.pp.log1p(sch_full)
sch_full = sch_full[ad_map.obs_names].copy()                 # align cells to the map
sc.pp.highly_variable_genes(sch_full, n_top_genes=2000)
hvg = list(sch_full.var_names[sch_full.var.highly_variable])
ad_ge = tg.project_genes(ad_map, sch_full[:, hvg].copy())    # MERFISH cells × HVG (imputed)
rna_imputed = np.asarray(ad_ge.X)
print("imputed RNA in space (MERFISH cells × HVG):", rna_imputed.shape)
pd.DataFrame(rna_imputed[:3, :5], columns=hvg[:5], index=adata_mer.obs_names[:3])

imputed RNA in space (MERFISH cells × HVG): (13864, 2000)


,Sox17,Gm37323,Gm37988,Rgs20,St18
index,,,,,
100019614084014214413331344619872059390,0.000021,0.000001,0.000143,0.274573,0.018893
100024126192764500990464846159908804485,0.000022,0.007023,0.000774,0.207134,0.097292
100036459431810739646600756615083690692,0.000021,0.000333,0.002274,0.184177,0.012309


## 12. Other omic in space ② — scATAC accessibility
Per-cell scATAC features were aggregated from the fragments file by the **same
18bp barcode** that keys the 3D genome (DNAbarcode), then carried into space via
the **same RNA-derived `ad_map`**. So ATAC is handled exactly like the 3D genome:
the spatial position is set by RNA; ATAC just rides along by cell id.

In [16]:
z = np.load(f"{ROOT}/data/schicar/atac_features.npz", allow_pickle=True)
atac = pd.DataFrame(z["binned_cpm"], index=z["rna_barcode"]).reindex(ad_map.obs_names)
A = atac.to_numpy(float); mA = np.isnan(A)
atac_spatial = (M.T @ np.where(mA, 0.0, A)) / (M.T @ (~mA).astype(float))   # MERFISH × 1Mb bins
nf = pd.Series(z["n_fragments"], index=z["rna_barcode"]).reindex(ad_map.obs_names).to_numpy()
mN = np.isnan(nf)
nf_spatial = (M.T @ np.where(mN, 0.0, nf)) / (M.T @ (~mN).astype(float))
print("ATAC 1Mb-binned accessibility in space (MERFISH cells × bins):", atac_spatial.shape)
print("ATAC depth n_fragments projected:", nf_spatial.shape)
print(">> projected through ad_map (the RNA-derived map): SPATIAL POSITION COMES FROM RNA.")

ATAC 1Mb-binned accessibility in space (MERFISH cells × bins): (13864, 2645)
ATAC depth n_fragments projected: (13864,)
>> projected through ad_map (the RNA-derived map): SPATIAL POSITION COMES FROM RNA.


## 13. Three scHiCAR modalities now in MERFISH space
All three are positioned by the single RNA mapping; 3D & ATAC are attached by cell id.

In [17]:
print("RNA imputed     :", rna_imputed.shape)
print("ATAC binned     :", atac_spatial.shape, " (+ atac_n_fragments", nf_spatial.shape, ")")
print("(paper 5kb loops added in the next section, §12b)")

RNA imputed     : (13864, 2000)
ATAC binned     : (13864, 2645)  (+ atac_n_fragments (13864,) )
(paper 5kb loops added in the next section, §12b)


## 12b. 3D-genome feature in space — the **paper's 5kb chromatin loops**
The 3D feature here is the **paper's own** identification: Peakachu 5kb chromatin
loops across 22 mouse-brain cell types (Nat. Biotechnol. s41587-026-03013-7,
Supplementary Table 5; 447,625 loops). For each scHiCAR cell we take its cell
type's **1Mb loop-anchor density**, then carry it into space via the same RNA
`ad_map` — exactly like ATAC. (We do NOT use our own computed 3D summaries.)

In [18]:
zL = np.load(f"{ROOT}/data/schicar/paper_loop_density_5kb.npz", allow_pickle=True)
loop_ct = list(zL["cell_types"]); loop_dens = zL["density"]          # (22, bins)
paper2sch = {"Pvalb":"Pvalb","Sst":"Sst","Vip":"Vip","PT":"PT","L45IT":"L45IT",
             "L5IT":"L5IT","L6IT":"L6IT","L6CT":"L6CT","L56NP":"L56NP","OPC":"OPC",
             "Oligo":"Oligo","MGL":"MGL","L23IT-1":"L23IT.1","L23IT-2":"L23IT.2",
             "L23IT-3":"L23IT.3","Astrocyte":"Astro","Endothelial":"Endo"}
sch_to_row = {paper2sch[p]: i for i, p in enumerate(loop_ct) if p in paper2sch}
cell_ct = sc.read_h5ad(f"{ROOT}/data/schicar/schicar_rna_mop.h5ad").obs["celltype"].astype(str)
cell_ct = cell_ct.reindex(ad_map.obs_names)
F_loop = np.full((len(cell_ct), loop_dens.shape[1]), np.nan, dtype=float)
for j, ct in enumerate(cell_ct):
    if ct in sch_to_row:
        F_loop[j] = loop_dens[sch_to_row[ct]]
mL = np.isnan(F_loop)
loop_spatial = (M.T @ np.where(mL, 0.0, F_loop)) / (M.T @ (~mL).astype(float))   # MERFISH × bins
print("paper 5kb loop-anchor density in space (MERFISH cells × 1Mb bins):", loop_spatial.shape)
print("scHiCAR types matched to paper loops:", sorted(sch_to_row))
print(">> per-cell = its cell type's loop density, carried to space via RNA ad_map.")

paper 5kb loop-anchor density in space (MERFISH cells × 1Mb bins): (13864, 2645)
scHiCAR types matched to paper loops: ['Astro', 'Endo', 'L23IT.1', 'L23IT.2', 'L23IT.3', 'L45IT', 'L56NP', 'L5IT', 'L6CT', 'L6IT', 'MGL', 'OPC', 'Oligo', 'PT', 'Pvalb', 'Sst', 'Vip']
>> per-cell = its cell type's loop density, carried to space via RNA ad_map.


## 14. Package into the U-Chrom `ChromData` format
ChromData natively stores imaging per-locus 3D coordinates; here we use it as a
**spatial multi-omic container** — each MERFISH cell is one `spot` (coords = its
centroid, z=0; `spots` are placeholders, NOT genomic loci), and the projected
omics go in `cells` + `cellm`. Provenance of the spatial positions is recorded in
`uns`.

In [19]:
import sys; sys.path.insert(0, "/Users/luzhang/Softwares/U-Chrom")
from uchrom.core import ChromData
xy = adata_mer.obsm["spatial"]; n = adata_mer.n_obs
ids = adata_mer.obs_names.to_numpy().astype(str)
coords = np.column_stack([xy[:, 0], xy[:, 1], np.zeros(n)]).astype(float)
spots = pd.DataFrame({"chrom": ["spatial"] * n, "start": np.arange(n),
                      "end": np.arange(n) + 1, "trace_id": ids, "cell_id": ids})
cells = pd.DataFrame({"x_centroid": xy[:, 0], "y_centroid": xy[:, 1],
                      "subclass": adata_mer.obs["subclass"].astype(str).values,
                      "pred_celltype": adata_mer.obs["pred_celltype"].astype(str).values,
                      "slice_id": adata_mer.obs["slice_id"].astype(str).values}, index=ids)
cellm = {"rna_imputed": rna_imputed.astype("float32"),
         "atac_binned": atac_spatial.astype("float32"),
         "atac_n_fragments": nf_spatial.reshape(-1, 1).astype("float32"),
         "loop_density_5kb": loop_spatial.astype("float32")}   # paper's 5kb loops in space
uns = {
  "spatial_source": "MERFISH MOp cell centroids (Zhang et al. 2021); cells positioned by "
                    "Tangram map_cells_to_space on shared RNA (mode=cells, density_prior=uniform)",
  "modality_provenance": "SPATIAL POSITION comes ONLY from scHiCAR-RNA -> MERFISH mapping. "
                    "3D-genome features and ATAC accessibility are per-cell scHiCAR measurements "
                    "(keyed by DNAbarcode), carried into space via the SAME RNA-derived ad_map; "
                    "they do NOT contribute positional information.",
  "note": "ChromData used as a spatial multi-omic container: each spot is a MERFISH-cell "
          "placeholder (coords = cell centroid, z=0); spots are NOT genomic loci.",
  "method": "tangram", "xyz_unit": "um",
  "threeD_features": "PAPER's 5kb chromatin loops (Peakachu), Nat Biotechnol "
                    "s41587-026-03013-7 Supplementary Table 5; 447,625 loops across 22 "
                    "cell types. cellm['loop_density_5kb'] = each cell's cell-type 1Mb "
                    "loop-anchor density, carried to space via the RNA ad_map. Raw loops: "
                    "data/schicar/paper_loops_5kb.csv. Our own computed 3D summaries are "
                    "NOT used. Types lacking paper loops (Peri, Lamp5) are NaN.",
  # genome-bin annotation for the binned cellm matrices (atac_binned, loop_density_5kb):
  "bin_chrom": np.array(z["bin_chrom"], dtype="S8"),       # bytes (decode with .astype(str))
  "bin_start": np.asarray(z["bin_start"], dtype=np.int64),
  "bin_size": 1000000,
  "bin_note": "1Mb genome bins (mm10 autosomes+X); these are the columns of "
              "cellm['atac_binned'] and cellm['loop_density_5kb']. coords/spots are "
              "per-cell placeholders (spatial, not genomic) — see 'note'.",
}
cd = ChromData(coords, spots, cells=cells, cellm=cellm, uns=uns)
cd

ChromData: n_spots=13864, n_traces=13864, n_cells=13864
  spots:   ['chrom', 'start', 'end', 'trace_id', 'cell_id']
  cells:   ['x_centroid', 'y_centroid', 'subclass', 'pred_celltype', 'slice_id'] (13864 cells)
  cellm:   {'rna_imputed': (13864, 2000), 'atac_binned': (13864, 2645), 'atac_n_fragments': (13864, 1), 'loop_density_5kb': (13864, 2645)}
  uns:     ['spatial_source', 'modality_provenance', 'note', 'method', 'xyz_unit', 'threeD_features', 'bin_chrom', 'bin_start', 'bin_size', 'bin_note']

## 15. Write the `.h5cd`

In [20]:
import os
H5CD = f"{ROOT}/data/schicar_to_merfish_spatial.h5cd"
cd.write(H5CD)
print("wrote", H5CD, "(%.1f MB)" % (os.path.getsize(H5CD) / 1e6))

wrote /Volumes/Extreme SSD/U-Chrom_explore_hicar/data/schicar_to_merfish_spatial.h5cd (407.3 MB)


## 16. Read it back with the U-Chrom reader and show the structure
`ChromData.read` is the U-Chrom feature for loading an `.h5cd`.

In [21]:
cd2 = ChromData.read(H5CD)
print(cd2)
print("\nn_spots:", cd2.n_spots, "| n_cells:", cd2.n_cells)
print("cellm:", {k: v.shape for k, v in cd2.cellm.items()})
cd2.cells.head()

ChromData: n_spots=13864, n_traces=13864, n_cells=13864
  spots:   ['chrom', 'start', 'end', 'trace_id', 'cell_id']
  cells:   ['x_centroid', 'y_centroid', 'subclass', 'pred_celltype', 'slice_id'] (13864 cells)
  cellm:   {'atac_binned': (13864, 2645), 'atac_n_fragments': (13864, 1), 'loop_density_5kb': (13864, 2645), 'rna_imputed': (13864, 2000)}
  uns:     ['bin_note', 'bin_size', 'method', 'modality_provenance', 'note', 'spatial_source', 'threeD_features', 'xyz_unit', 'bin_chrom', 'bin_start']

n_spots: 13864 | n_cells: 13864
cellm: {'atac_binned': (13864, 2645), 'atac_n_fragments': (13864, 1), 'loop_density_5kb': (13864, 2645), 'rna_imputed': (13864, 2000)}


,x_centroid,y_centroid,subclass,pred_celltype,slice_id
100019614084014214413331344619872059390,-4406.866048,-3784.263422,L4/5 IT,L45IT,mouse2_slice61
100024126192764500990464846159908804485,-5938.214987,1334.835803,L4/5 IT,L5IT,mouse2_slice1
100036459431810739646600756615083690692,1207.536506,2999.896805,L4/5 IT,L45IT,mouse2_slice300
100036464372885187596454120358701331853,2121.430284,2490.716746,L5 ET,PT,mouse2_slice300
100042375868071099085129438220019302596,-4863.292523,-3079.352587,Oligo,Oligo,mouse2_slice61


In [22]:
# spatial-position provenance recorded in the file:
print("spatial_source      :", cd2.uns["spatial_source"])
print("\nmodality_provenance :", cd2.uns["modality_provenance"])
print("\nnote                :", cd2.uns["note"])
bc = np.asarray(cd2.uns["bin_chrom"]).astype(str)
print("\nbin_note            :", cd2.uns["bin_note"])
print("genome-bin annotation for atac_binned / loop_density_5kb:", len(bc), "bins; e.g.",
      list(zip(bc[:3], cd2.uns["bin_start"][:3])), "... ", bc[-1], int(cd2.uns["bin_start"][-1]))
cd2.spots.head()

spatial_source      : MERFISH MOp cell centroids (Zhang et al. 2021); cells positioned by Tangram map_cells_to_space on shared RNA (mode=cells, density_prior=uniform)

modality_provenance : SPATIAL POSITION comes ONLY from scHiCAR-RNA -> MERFISH mapping. 3D-genome features and ATAC accessibility are per-cell scHiCAR measurements (keyed by DNAbarcode), carried into space via the SAME RNA-derived ad_map; they do NOT contribute positional information.

note                : ChromData used as a spatial multi-omic container: each spot is a MERFISH-cell placeholder (coords = cell centroid, z=0); spots are NOT genomic loci.

bin_note            : 1Mb genome bins (mm10 autosomes+X); these are the columns of cellm['atac_binned'] and cellm['loop_density_5kb']. coords/spots are per-cell placeholders (spatial, not genomic) — see 'note'.
genome-bin annotation for atac_binned / loop_density_5kb: 2645 bins; e.g. [('chr1', 0), ('chr1', 1000000), ('chr1', 2000000)] ...  chrX 171000000


,chrom,start,end,trace_id,cell_id
0,spatial,0,1,100019614084014214413331344619872059390,100019614084014214413331344619872059390
1,spatial,1,2,100024126192764500990464846159908804485,100024126192764500990464846159908804485
2,spatial,2,3,100036459431810739646600756615083690692,100036459431810739646600756615083690692
3,spatial,3,4,100036464372885187596454120358701331853,100036464372885187596454120358701331853
4,spatial,4,5,100042375868071099085129438220019302596,100042375868071099085129438220019302596


## 17. Honest scope notes
- **3D feature = the paper's own 5kb loops** (Peakachu, 22 cell types). The `.h5cd`
  now carries RNA + ATAC + paper-loop density. We deliberately do **not** use our
  own computed 3D summaries.
- The loops are **per cell type** (pseudobulk); each spatial cell inherits its
  predicted cell type's loop-anchor density — so this is cell-type-resolution.
  Two of our types (Peri, Lamp5) have no paper loops → NaN.
- **Spatial positions come only from the RNA mapping** (Tangram). ATAC and the
  paper loops are attached by cell id and carried to space by that mapping — they
  add no positional information.
- **ChromData is used here as a spatial multi-omic container**; its `coords`/`spots`
  are per-cell placeholders, not the imaging per-locus 3D coordinates the format was
  designed for.
- ATAC depth here is ~30k fragments/cell (good); still, all imputed-in-space values
  inherit the RNA bridge's cell-type-level resolution (see project validation).